# XANDAR — recortes NISAR com NASA Harmony

**Objetivo:** testar um job pequeno de 19/08/2026 (somente HHHH) e, depois, solicitar explicitamente HHHH, HVHV e máscara para 19/08 e 31/08/2026. A saída esperada é COG/GeoTIFF; o notebook não contém um caminho de download do HDF5 original.

**Estado da evidência (14/11/2026):** capacidades públicas da coleção confirmadas em consulta real. Autenticação Bearer, requisição e assinatura de `harmony-py` verificadas na documentação e no código oficiais. **Nenhum job autenticado foi executado pelo autor deste notebook.** Tempo, bytes finais e suporte específico dos dois granules dependem da execução abaixo.

Use um ambiente **CPU**. Não é necessário instalar GDAL local para solicitar o processamento no servidor Harmony. O job permanece no servidor se o navegador fechar; os identificadores são gravados imediatamente em `manifesto_harmony.json`, permitindo retomar o acompanhamento sem reenviar o mesmo pedido.

O retângulo proposto é de aproximadamente **10 × 10 km**, centrado perto da origem do colapso. Não é o polígono mapeado do desastre e não abrange toda a inundação. Mantemos a grade nativa: não pedimos reprojeção, interpolação, largura nem altura.

## Segurança e resultados

- Insira somente seu próprio token EDL no campo oculto. Não o cole no código nem em mensagens.
- O manifesto não guarda token, senha, cabeçalhos nem URLs assinadas de saída.
- O disco `/content` do Colab é temporário. Para sobreviver à exclusão do ambiente, salve o manifesto e resultados no Drive (opção abaixo) ou baixe os checkpoints. O notebook não pode recuperar arquivos locais apagados pelo Colab.
- Não subtraia rasters com grades diferentes. O notebook apenas informa a compatibilidade; não os realinha automaticamente.
- Dados **PROVISIONAL**, com limitações de processamento. Ler a máscara e a documentação de qualidade antes de interpretar diferenças.

Referências: [NISAR Harmony](https://nisar-docs.asf.alaska.edu/harmony/), [API Harmony e Bearer](https://harmony.earthdata.nasa.gov/docs), [harmony-py](https://github.com/nasa/harmony-py), [problemas conhecidos](https://nisar-docs.asf.alaska.edu/nisar-known-issues/).


## 1. Instalar bibliotecas
Instala bibliotecas do PyPI; as versões realmente utilizadas serão registradas no manifesto.

In [ ]:
%pip -q install -U harmony-py rasterio numpy requests


## 2. Destino, área e cenas

Por padrão, os resultados ficam em `/content/XANDAR_Harmony`. Para persistir no Drive, descomente as duas linhas de montagem e altere `RESULT_DIR` para uma pasta sua no Drive. A autorização do Drive é feita pela interface Google; o token NASA nunca é gravado nessa pasta.

Para retomar um job existente, mantenha o mesmo `RESULT_DIR` com seu manifesto. Se restaurar apenas o manifesto em outro ambiente, os TIFFs serão baixados novamente a partir do job, se ainda disponíveis. Não há garantia de retenção indefinida das saídas do Harmony.


In [ ]:
from pathlib import Path
import os, json, time, hashlib, re, inspect
from datetime import datetime, timezone
from urllib.parse import urlsplit, unquote
import importlib.metadata as im
import requests
import numpy as np
import rasterio

# Opcional: persistência no Drive. Ative antes de criar o cliente Harmony.
# from google.colab import drive
# drive.mount('/content/drive')

RESULT_DIR = Path('/content/XANDAR_Harmony')
# RESULT_DIR = Path('/content/drive/MyDrive/XANDAR_Harmony')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = RESULT_DIR / 'manifesto_harmony.json'

COLLECTION = 'NISAR_L2_GCOV_PROVISIONAL_V1'
COLLECTION_ID = 'C2854338529-ASF'
BBOX = (85.4772, 28.2437, 85.5792, 28.3337)  # oeste, sul, leste, norte
SCENES = {
    '2026-08-19': 'NISAR_L2_PR_GCOV_028_098_A_016_4005_DHDH_A_20260819T233918_20260819T233953_X05026_N_F_J_001',
    '2026-08-31': 'NISAR_L2_PR_GCOV_029_098_A_016_4005_DHDH_A_20260831T233918_20260831T233952_X05026_N_F_J_001',
}
VARIABLES = {
    'HHHH': '/science/LSAR/GCOV/grids/frequencyA/HHHH',
    'HVHV': '/science/LSAR/GCOV/grids/frequencyA/HVHV',
    'mask': '/science/LSAR/GCOV/grids/frequencyA/mask',
}

def now():
    return datetime.now(timezone.utc).isoformat()

def save_manifest():
    tmp = MANIFEST_PATH.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    tmp.replace(MANIFEST_PATH)

if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    if manifest.get('collection') != COLLECTION or tuple(manifest.get('bbox_wgs84', [])) != BBOX:
        raise RuntimeError('Manifesto pertence a outra coleção/área. Escolha outra pasta; não o sobrescreva.')
else:
    manifest = {
        'created_utc': now(), 'collection': COLLECTION, 'collection_id': COLLECTION_ID,
        'bbox_wgs84': list(BBOX), 'scenes': SCENES,
        'aoi_note': 'Retângulo proposto ~10 × 10 km ao redor da origem; não é a delimitação do desastre.',
        'jobs': {}, 'files': [], 'grid_checks': {},
        'evidence': 'Capacidades consultadas; extração autenticada depende desta execução.',
    }
manifest['versions'] = {p: im.version(p) for p in ['harmony-py', 'rasterio', 'numpy', 'requests']}
save_manifest()
print('Destino:', RESULT_DIR)
print('Jobs já registrados:', len(manifest['jobs']))
print('Versões:', manifest['versions'])


## 3. Testar as capacidades públicas da coleção

Esta célula não pede token, não submete jobs e não baixa cenas. Exige suporte efetivo a bbox, variável e TIFF no mesmo serviço científico. A lista de variáveis da coleção pode incluir campos ausentes em um granule; o serviço pode ignorá-los. Por isso conferimos as saídas após cada job.


In [ ]:
CAPABILITIES_URL = 'https://harmony.earthdata.nasa.gov/capabilities'
response = requests.get(CAPABILITIES_URL, params={'shortName': COLLECTION}, timeout=(20, 60))
response.raise_for_status()
capabilities = response.json()
if capabilities.get('conceptId') != COLLECTION_ID:
    raise RuntimeError('Identificador da coleção mudou; revisar a configuração antes de submeter jobs.')
services = capabilities.get('services', [])
scientific = [s for s in services if s.get('name') == 'nasa/harmony-gdal-adapter']
if len(scientific) != 1:
    raise RuntimeError('Serviço científico Harmony não encontrado. Não foi iniciado download.')
service_caps = scientific[0].get('capabilities', {})
subset = service_caps.get('subsetting', {})
formats = {f.get('mimeType') for f in service_caps.get('outputFormats', [])}
available_variables = {v['name'] for v in capabilities.get('variables', [])}
if not subset.get('bbox') or not subset.get('variable') or 'image/tiff' not in formats:
    raise RuntimeError('A coleção não declara a combinação bbox + variável + TIFF.')
if not set(VARIABLES.values()).issubset(available_variables):
    raise RuntimeError('Faltam variáveis esperadas nas capacidades. Revisar antes de prosseguir.')
(RESULT_DIR/'capabilities.json').write_text(json.dumps(capabilities, ensure_ascii=False, indent=2), encoding='utf-8')
manifest['capabilities_checked_utc'] = now()
manifest['scientific_service'] = scientific[0]['name']
save_manifest()
print('Confirmado: bbox + seleção de variáveis + image/tiff.')
print('Polígono arbitrário suportado:', bool(subset.get('shape')))
print('O fluxo utilizará bbox, saída nativa e somente variáveis selecionadas.')


## 4. Autenticar com token EDL

Execute a célula e cole o token no campo oculto, sem `Bearer`. Isso testa a autenticação no Harmony, **não** a compatibilidade com download direto do HDF5. O processamento científico é feito pelo serviço servidor.

Se a conta tiver termos pendentes ou não autorizar a aplicação, complete o procedimento no portal EDL. Não inclua o token na mensagem de erro ao pedir ajuda.


In [ ]:
os.environ['NUM_REQUESTS_WORKERS'] = '1'
from harmony import Client, Request, Collection, BBox
from getpass import getpass

required_client = {'token', 'download_timeout'}
required_request = {'spatial', 'variables', 'granule_name', 'format', 'max_results', 'ignore_errors'}
if not required_client.issubset(inspect.signature(Client).parameters):
    raise RuntimeError('Versão instalada do harmony-py não possui a assinatura esperada. Reinicie após atualizar.')
if not required_request.issubset(inspect.signature(Request).parameters):
    raise RuntimeError('Assinatura Request diferente da verificada. Não submeter automaticamente.')

edl_token = getpass('Token EDL (entrada oculta, sem Bearer): ').strip()
if not edl_token or any(c.isspace() for c in edl_token):
    edl_token = None
    raise RuntimeError('Token vazio ou contém espaços. Cole apenas o token.')
try:
    client = Client(token=edl_token, check_interval=10.0, download_timeout=(20, 120))
except Exception:
    raise RuntimeError('Autenticação Harmony falhou. Verifique validade do token, aplicação e termos EDL. Nenhum job foi submetido.') from None
finally:
    edl_token = None
print('Cliente Harmony configurado com token. A aceitação será verificada ao submeter o job; o token permanece somente na memória privada do cliente.')


## 5. Funções de acompanhamento, download e inspeção

Cada job é registrado **imediatamente após a submissão**. Reexecutar a célula de um experimento procura o job equivalente no manifesto e o acompanha, sem enviar um pedido duplicado. O acompanhamento espera no máximo 45 minutos por execução; o job continua no Harmony, com ID preservado.

Só baixamos URLs de saída fornecidas pelo job, com extensão TIFF. Não há fallback para links `.h5`. Alguns resultados podem ter tamanho maior que a estimativa: variáveis, resolução nativa, compressão e máscaras influenciam o tamanho. Inspecione o primeiro experimento antes da expansão.


In [ ]:
def safe_tags(tags):
    # Do not persist credential-like metadata or signed URLs.
    forbidden = re.compile(r'token|password|authorization|signature|credential|secret', re.I)
    return {str(k): str(v) for k, v in tags.items()
            if not forbidden.search(str(k)) and not forbidden.search(str(v))}

def serial_nodata(v):
    if v is None:
        return None
    if isinstance(v, (float, np.floating)) and not np.isfinite(v):
        return str(v)
    return v.item() if hasattr(v, 'item') else v

def infer_layer(path, tags, requested):
    text = unquote(path.name) + ' ' + json.dumps(tags)
    found = [name for name in VARIABLES
             if re.search(r'(^|[/_.\s-])' + re.escape(name) + r'([/_.\s-]|$)', text, re.I)]
    if len(found) == 1:
        return found[0]
    return None

def inspect_tiff(path, date, job_id, requested):
    with rasterio.open(path) as ds:
        if ds.driver != 'GTiff' or ds.crs is None or ds.width == 0 or ds.height == 0:
            raise RuntimeError('Saída não é um GeoTIFF georreferenciado válido.')
        tags = safe_tags(ds.tags())
        structure = safe_tags(ds.tags(ns='IMAGE_STRUCTURE'))
        info = {
            'file': str(path.relative_to(RESULT_DIR)), 'date_utc':date, 'job_id':job_id,
            'layer':infer_layer(path, tags, requested), 'requested_layers':requested,
            'bytes':path.stat().st_size, 'driver':ds.driver, 'dtype':list(ds.dtypes),
            'crs':ds.crs.to_string(), 'epsg':ds.crs.to_epsg(),
            'transform':list(ds.transform)[:6], 'width':ds.width, 'height':ds.height,
            'resolution':list(ds.res), 'bands':ds.count, 'nodata':[serial_nodata(x) for x in ds.nodatavals],
            'tiled':bool(ds.profile.get('tiled')), 'block_shapes':[list(x) for x in ds.block_shapes],
            'image_structure':structure, 'cog_layout_declared':structure.get('LAYOUT') == 'COG',
            'overviews':ds.overviews(1), 'tags':tags,
            'mask_flags':[[flag.name for flag in flags] for flags in ds.mask_flag_enums],
        }
        # Tiny readability check, not a scientific validation or full scan.
        sample = ds.read(1, window=rasterio.windows.Window(0, 0, min(32, ds.width), min(32, ds.height)))
        info['sample_readable'] = sample.size > 0
    with path.open('rb') as source:
        info['sha256'] = hashlib.file_digest(source, 'sha256').hexdigest()
    print(f"  {info['layer'] or 'CAMADA NÃO IDENTIFICADA'}: {info['width']} × {info['height']}, "
          f"{info['dtype']}, {info['crs']}, {info['bytes']/1024**2:.2f} MiB; COG declarado={info['cog_layout_declared']}")
    return info

def key_for(date, layers):
    specification = json.dumps({'scene':SCENES[date], 'bbox':BBOX, 'layers':sorted(layers),
                                'format':'image/tiff'}, sort_keys=True)
    return hashlib.sha256(specification.encode()).hexdigest()[:20]

def wait_job(job, max_seconds=45*60):
    started = time.monotonic()
    while True:
        progress, status, _ = client.progress(job['id'])
        job.update(status=status, progress=int(progress), checked_utc=now())
        save_manifest()
        print(f"Job {job['id']}: {status}, {progress}%", end='\r')
        if status == 'successful':
            print()
            return
        if status in {'failed','canceled','paused','complete_with_errors'}:
            print()
            raise RuntimeError(f"Job em estado {status}. ID preservado no manifesto; revise no Harmony antes de reenviar.")
        if time.monotonic() - started > max_seconds:
            print()
            raise RuntimeError('Tempo local de acompanhamento terminou. Execute novamente para retomar o mesmo job; ele não foi cancelado.')
        time.sleep(10)

def run_job(date, layers):
    if not set(layers).issubset(VARIABLES):
        raise ValueError('Camada desconhecida.')
    key = key_for(date, layers)
    job = manifest['jobs'].get(key)
    if job is None:
        request = Request(
            Collection(id=COLLECTION_ID), spatial=BBox(*BBOX),
            variables=[VARIABLES[name] for name in layers], granule_name=[SCENES[date]],
            format='image/tiff', max_results=1, ignore_errors=False,
        )
        if not request.is_valid():
            raise RuntimeError('Requisição inválida: ' + str(request.error_messages()))
        job_id = client.submit(request)
        job = {'id':str(job_id), 'date_utc':date, 'scene':SCENES[date], 'layers':layers,
               'submitted_utc':now(), 'status':'submitted', 'outputs':[]}
        manifest['jobs'][key] = job
        save_manifest()  # Before polling or downloading.
        print('Job submetido e salvo:', job['id'])
    else:
        print('Retomando job registrado:', job['id'])
    wait_job(job)
    # These are transformation outputs from this job, never the original ASF URLs.
    urls = list(client.result_urls(job['id']))
    if not urls:
        raise RuntimeError('Job não retornou saídas. Verifique disponibilidade das variáveis.')
    for url in urls:
        parsed = urlsplit(url)
        suffix = Path(unquote(parsed.path)).suffix.lower()
        if parsed.scheme != 'https' or suffix not in {'.tif','.tiff'}:
            raise RuntimeError('Job retornou saída que não é TIFF HTTPS. Download não iniciado; revisar no Harmony.')
    directory = RESULT_DIR/date/key
    directory.mkdir(parents=True, exist_ok=True)
    # Download Futures are awaited individually. No full source-file download exists here.
    infos=[]
    for url in urls:
        try:
            path = Path(client.download(url, directory=str(directory), overwrite=True).result())
        except Exception:
            raise RuntimeError('Download da saída Harmony falhou. Job ID está salvo; reexecute para tentar novamente. Nenhum HDF5 original foi solicitado.') from None
        info = inspect_tiff(path, date, job['id'], layers)
        infos.append(info)
        # Persist after each completed file, not only at the end of a batch.
        manifest['files'] = [f for f in manifest['files'] if f['file'] != info['file']] + [info]
        job['outputs'] = [f['file'] for f in infos]
        save_manifest()
    found = {f['layer'] for f in infos}
    missing = sorted(set(layers) - found)
    job['missing_layers'] = missing
    job['outputs_inspected_utc'] = now()
    save_manifest()
    if missing:
        print('ATENÇÃO: camadas não confirmadas nas saídas:', missing)
        print('O serviço pode ignorar variáveis ausentes. Não concluir que a máscara foi obtida sem identificá-la.')
    return infos

def latest_file(date, layer):
    matches = [f for f in manifest['files'] if f['date_utc']==date and f.get('layer')==layer
               and (RESULT_DIR/f['file']).is_file()]
    return matches[-1] if matches else None

def compare_grids(first, second):
    return (first['crs']==second['crs'] and first['width']==second['width']
            and first['height']==second['height']
            and bool(np.allclose(first['transform'], second['transform'], rtol=0, atol=1e-5)))


## 6. Primeiro teste — somente 19/08, HHHH

Submete um único granule, uma única variável, bbox de aproximadamente 10 × 10 km. Confirme a saída, dimensões, resolução, CRS e tamanho antes de executar a expansão. O término do job valida este fluxo para esse pedido; não demonstra qualidade científica dos pixels.

Uma variável Float32 em 10 × 10 km teria aproximadamente **4 MB a 10 m**, sem compressão nem overhead. É uma estimativa dimensional, não o tamanho observado; a resolução real será informada pela inspeção.


In [ ]:
first_test = run_job('2026-08-19', ['HHHH'])
if not any(info.get('layer') == 'HHHH' for info in first_test):
    raise RuntimeError('HHHH não foi identificada. Não execute a expansão antes de revisar.')
print('Teste inicial concluído. Revise os metadados e o tamanho acima.')
print('Manifesto:', MANIFEST_PATH)


## 7. Checkpoint opcional — salvar o manifesto

Execute para baixar o manifesto antes de continuar, especialmente se estiver usando `/content`. Ele registra IDs, pedidos e saídas, sem credenciais. Não evita que o Colab apague os TIFFs: para mantê-los, use o Drive ou baixe o ZIP final.


In [ ]:
from google.colab import files
files.download(str(MANIFEST_PATH))


## 8. Expansão explícita — três variáveis nas duas datas

**Execute somente depois de revisar o teste inicial.** Esta célula reutiliza HHHH de 19/08 se o arquivo existir e pede apenas HVHV + máscara restantes. Em seguida pede HHHH, HVHV e máscara de 31/08, sequencialmente. Se a primeira saída não existir, para e pede para retomar o teste; não começa um download integral.

Para seis rasters (Float32 + Float32 + Byte em cada data) de 10 × 10 km a 10 m, o volume bruto de pixels seria aproximadamente **18 MB (~20 MB com alguma margem)**. Não é promessa de tamanho: compressão, resolução nativa, alpha, nodata e metadados podem alterar esse valor. O processamento ocorre no servidor; seu uso interno do HDF5 não é o volume transferido ao Colab.


In [ ]:
if latest_file('2026-08-19', 'HHHH') is None:
    raise RuntimeError('O teste inicial não está salvo. Retome a célula 6 antes da expansão.')

for date in SCENES:
    missing = [name for name in ['HHHH','HVHV','mask'] if latest_file(date, name) is None]
    if missing:
        print('Solicitando', date, missing)
        run_job(date, missing)
    else:
        print(date, ': três camadas já salvas; sem novo pedido.')

print('Expansão terminada. Execute a verificação de completude e grades.')


## 9. Verificar completude, grades e tamanhos

Não faz reprojeção nem realinhamento. Se grades diferirem, fica registrado e a comparação pixel a pixel deve parar. Mesmo com grades coincidentes, interprete os códigos da máscara e os problemas conhecidos dos produtos PROVISIONAL antes de qualquer análise temporal.


In [ ]:
checks = {}
all_complete = True
for layer in ['HHHH','HVHV','mask']:
    a, b = [latest_file(date, layer) for date in SCENES]
    if a is None or b is None:
        checks[layer] = {'complete':False, 'same_grid':None}
        all_complete = False
        print(layer, ': AUSENTE/NÃO IDENTIFICADA em pelo menos uma data.')
    else:
        same = compare_grids(a,b)
        checks[layer] = {'complete':True, 'same_grid':same,
                         'before':a['file'], 'after':b['file']}
        print(layer, ': grades coincidentes =', same)
        if not same:
            print('  PARAR: não subtrair nem comparar pixels antes de alinhar explicitamente.')

# Also compare the mask to the science layer within each date.
for date in SCENES:
    reference = latest_file(date, 'HHHH')
    for layer in ['HVHV','mask']:
        candidate = latest_file(date, layer)
        checks[date+'_'+layer] = {'same_grid': compare_grids(reference,candidate)
                                  if reference and candidate else None}

manifest['grid_checks'] = checks
manifest['all_six_layers_present'] = all_complete
manifest['verified_utc'] = now()
save_manifest()
chosen = [latest_file(date,layer) for date in SCENES for layer in ['HHHH','HVHV','mask']]
total = sum(item['bytes'] for item in chosen if item)
print(f'Tamanho dos arquivos confirmados: {total/1024**2:.2f} MiB')
print('Seis camadas presentes:', all_complete)
print('COG declarado em todos os arquivos:', all(item['cog_layout_declared'] for item in chosen if item) if all_complete else 'incompleto')
print('A presença LAYOUT=COG é uma declaração do driver; não substitui validação formal da especificação COG.')


## 10. Baixar os COGs e o manifesto

O ZIP usa apenas os arquivos de saída confirmados por data/camada, o manifesto e as capacidades consultadas. Não inclui credenciais. Um conjunto incompleto pode ser baixado para diagnóstico, mas será nomeado como incompleto e não deve ser tratado como resultado final.


In [ ]:
import zipfile
complete = manifest.get('all_six_layers_present',False)
zip_path = RESULT_DIR / ('XANDAR_Harmony_recortes.zip' if complete else 'XANDAR_Harmony_INCOMPLETO.zip')
selected = [latest_file(date, layer) for date in SCENES for layer in ['HHHH','HVHV','mask']]
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_STORED) as z:
    z.write(MANIFEST_PATH, MANIFEST_PATH.name)
    z.write(RESULT_DIR/'capabilities.json','capabilities.json')
    for item in selected:
        if item:
            z.write(RESULT_DIR/item['file'], item['file'])
print(f'Pacote: {zip_path.stat().st_size/1024**2:.2f} MiB; completo={complete}')
files.download(str(zip_path))


## 11. Encerrar o cliente e remover referências ao token

Execute ao terminar. Isso não revoga o token na NASA; apenas fecha a sessão e remove suas referências no cliente desta execução. Cada integrante deve usar seu próprio token e ambiente. Não redistribua o ambiente de execução com o cliente autenticado.


In [ ]:
if 'client' in globals():
    if getattr(client,'session',None) is not None:
        client.session.close()
    client.token = None
    client.auth = None
    if hasattr(client,'executor'):
        client.executor.shutdown(wait=True)
    del client
edl_token = None
print('Cliente encerrado. Resultados e manifesto preservados no destino escolhido.')
